In [ ]:
#Importing relevant Python libraries and modules
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#A function for extracting the electrostatic interaction energetic values from .csv files stored within a directory of interest
#Input: a directory of interest, containing electrostatic interaction energy files in .csv format; the number of copies of the tested system
#Output: a dictionary containing all energetic values, with the keys depicting the two interacting agents (typically, an Hfq region and polyphosphate/DNA) and the values constituting the interaction energies
def get_file_name_components(dir, copy_nums):
    
    #Storing all .csv energy files and defining the output dictionary
    file_list = os.listdir(dir)
    file_list_csv_only = [x for x in file_list if x.endswith('.csv')]
    output_dict = {}

    #Running on each energy file and extracting the energy values, storing them with the relevant key in the output dictionary
    for file in file_list_csv_only:
        first_split = file.split('_')
        last_split_term = first_split[-1]
        copy_serial = last_split_term.split('.')[0]
        if len(first_split[0]) > 1:
            if len(first_split[1]) > 1:
                curr_key = first_split[0] + '-' + first_split[1]
            else:
                curr_key = first_split[0] + '-' + first_split[1] + '-' + first_split[2]
        elif len(first_split[2]) > 1:
            curr_key = first_split[0] + '-' + first_split[1] + '-' + first_split[2]
        else:
            curr_key = first_split[0] + '-' + first_split[1] + '-' + first_split[2] + '-' + first_split[3]
        file_df = pd.read_csv(dir + file, sep = '\t')
        file_df.columns = ['Energy']
        if int(copy_serial) in copy_nums:
            if curr_key in output_dict:
                output_dict[curr_key].append(file_df)
            else:
                output_dict[curr_key] = [file_df]

    #Returning the output dictionary
    return output_dict

#Defining the directories containing the energy files of the relevant CG system
origin_dir_CURR_SYSTEM = '' ####PASTE THE DIRECTORY OF ENERGY FILES HERE
copy_number_CURR_SYSTEM = 0 ####PASTE THE NUMBER OF SIMULATION COPIES HERE
CURR_SYSTEM_copies = [i + 1 for i in range(copy_number_CURR_SYSTEM)]

#Extracting the energy values of each system to a corresponding output dictionary
CURR_SYSTEM_dfs_dict = get_file_name_components(origin_dir_CURR_SYSTEM, CURR_SYSTEM_copies)

In [ ]:
#Defining all keys from all systems, namely all pairs of interacting agents (typically, an Hfq region and polyphosphate/DNA)
all_keys_CURR_SYSTEM = [key for key in (list(CURR_SYSTEM_dfs_dict.keys()))]
all_keys_set_CURR_SYSTEM = list(set(all_keys_CURR_SYSTEM))

In [ ]:
#Filtering out first 10% energetic values of all systems for proper energetic sampling
filter_thresh = 1000
dicts = [CURR_SYSTEM_dfs_dict]
unified_dicts = [{}]
for key in all_keys_set_CURR_SYSTEM:
    for i_d, d in enumerate(dicts):
        if key in d:
            curr_dfs_list = d[key]
            curr_unified_df = pd.DataFrame(columns = ['Energy'])
            for df in curr_dfs_list:
                filtered_df = df.copy().iloc[filter_thresh:, :].reset_index()
                curr_unified_df = pd.concat([curr_unified_df, filtered_df], axis = 0, ignore_index = True)
            unified_dicts[i_d][key] = curr_unified_df

In [ ]:
#Defining parameters relevant for calculating electrostatic Debye-Huckle energetic term
K_coulomb = 332 * 4.184 #kJ/mol*e^2
ionic_strength = 0.02
ref_dielectric_constant = 78.5
dielectric_constant = 70
ionic_radius = 1.4 #A
screening_factor = 0.33 * (ref_dielectric_constant / dielectric_constant) * np.sqrt(ionic_strength) #1/A
salt_dependent_coefficient = (np.e ** (screening_factor * ionic_radius)) / (1 + (screening_factor * ionic_radius))

#Getting a scaling factor to multiply by the raw energy values for getting correct units (kcal/mol)
factor_to_multiply_by = K_coulomb * salt_dependent_coefficient * (10 ** 10)

In [ ]:
#Generating 2D KDE plots of electrostatic interaction energies, comparing each relevant pair of interacting agents with another (relevant, in that case, means the interaction of the distal and proximal Hfq's faces with polyphosphate or DNA)
#IMPORTANT: IN CASE ONE WOULD LIKE TO GENERATE THE SAME PLOTS AS IN THE FIGURES, THERE IS A NEED TO SUM UP THE ENERGETIC CONTRIBUTIONS OF THE LATERAL (RIM) AND PROXIMAL FACES IN CASE PRESENTED, OR SUM UP ALL FACES OF HFQ IN THE SIMULATIONS OF 2 PROTEINS TO GET THE TOTAL HFQ INTERACTION

#Getting the relevant pairs of interacting agents for each system type
smaller_keys_set = all_keys_set_CURR_SYSTEM
titles = ['PASTE YOUR TITLE HERE']
for i_first_key in range(0, len(smaller_keys_set)):
    for i_second_key in range(i_first_key + 1, len(smaller_keys_set)):
        first_key = smaller_keys_set[i_first_key]
        second_key = smaller_keys_set[i_second_key]

        #For each comparison of the two relevant pairs, producing the 2D KDE plot as PDF file and saving it
        for k in range(len(unified_dicts)):
            if (first_key in list(unified_dicts[k].keys())) and (second_key in list(unified_dicts[k].keys())):
                if (len(np.unique(unified_dicts[k][first_key]['Energy'])) > 1) and (len(np.unique(unified_dicts[k][second_key]['Energy'])) > 1):
                    plt.rcParams['pdf.fonttype'] = 42
                    fig, ax = plt.subplots(figsize = (8, 6))
                    sns.kdeplot(ax = ax, x = unified_dicts[k][first_key]['Energy'] * factor_to_multiply_by, y = unified_dicts[k][second_key]['Energy'] * factor_to_multiply_by, log_scale = False, shade = True, cbar = True, color = 'lightskyblue', label = titles[k], alpha = 1)
                    ax.set_xlabel(first_key, fontsize = 20)
                    ax.set_ylabel(second_key, fontsize = 20)
                    lims = [np.min([ax.get_xlim(), ax.get_ylim()]), np.max([ax.get_xlim(), ax.get_ylim()])]
                    ax.set_xlim(lims)
                    ax.set_ylim(lims)
                    ax.set_title(titles[k])
                    ax.plot(lims, lims, '--k', zorder = 1, alpha = 0.65)
                    fig.savefig(origin_dir_CURR_SYSTEM + '2D-KDE-Plot_' + first_key + '&' + second_key + '_' + titles[k] + '.pdf', format = 'pdf', dpi = 300)

In [ ]:
#Generating 1D KDE plots of electrostatic interaction energies for each pair of interacting agents, of each system of interest
dict_names = ['PASTE THE SYSTEM OF INTEREST HERE']
unified_dicts_1D = [unified_dicts]
for i_dict, curr_dict in enumerate(unified_dicts_1D):
    curr_keys = list(curr_dict[0].keys())
    for curr_key in curr_keys:
        plt.rcParams['pdf.fonttype'] = 42
        fig, ax = plt.subplots(figsize = (5, 5))
        sns.kdeplot(curr_dict[0][curr_key].copy()['Energy'] * factor_to_multiply_by, c = 'brown', linewidth = 1)
        ax.set_title(dict_names[i_dict] + ' - ' + curr_key)
        fig.savefig(origin_dir_CURR_SYSTEM + '1D-KDE-Plot_' + dict_names[i_dict] + '_' + curr_key + '.pdf', format = 'pdf', dpi = 300)